# Day 3: Deep Learning using CNN (Smart Agriculture)

## 01 Problem Statement

**Business Problem:**
The agricultural industry is heavily reliant on manual labor for harvesting crops. This process is time-consuming, expensive, and prone to human error, often leading to wasted produce. Automating the harvesting process using robotic arms and computer vision can significantly optimize yield and reduce costs.

**Why CNNs over Handcrafted ML Features?**
In Days 1 and 2, we used handcrafted features (like mean RGB/HSV and color histograms) paired with traditional ML algorithms. While effective to a degree, this approach lacks **spatial awareness** and relies heavily on human intuition to engineer features. 
Convolutional Neural Networks (CNNs) automate feature extraction. They apply spatial filters across the image to learn hierarchical patterns (edges, textures, shapes), making them significantly more robust to variations in lighting, background noise, and orientation without requiring rigid handcrafted features.

**Deep Learning Problem Definition:**
This project focuses on a **Binary Image Classification** task. The goal is to build a custom Convolutional Neural Network from scratch to automatically extract spatial features and classify tomato images into two distinct classes: **Ripe** and **Unripe**.

## 02 Imports
Importing all necessary libraries for data processing, model building, and evaluation.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import random

# Deep Learning Libraries (TensorFlow & Keras)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Machine Learning Evaluation Metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report,
                             roc_curve, precision_recall_curve)

# Ensure reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

print(f"TensorFlow Version: {tf.__version__}")

## 03 Dataset Loading
- Load images directly from `Dataset/detection/images/`
- Automatically detect classes based on filenames (riped vs unriped)
- Display dataset summary and class distribution
- Show sample images

In [ ]:
# Define Dataset Path
DATASET_PATH = 'Dataset/detection/images/'

# Automatically detect classes from filenames
filepaths = []
labels = []
class_names = ['unripe', 'ripe'] # Assuming unriped is 0, riped is 1

if os.path.exists(DATASET_PATH):
    for filename in os.listdir(DATASET_PATH):
        if filename.endswith('.jpeg') or filename.endswith('.jpg') or filename.endswith('.png'):
            filepaths.append(os.path.join(DATASET_PATH, filename))
            if 'unriped' in filename.lower():
                labels.append(0) # Unripe
            elif 'riped' in filename.lower():
                labels.append(1) # Ripe

# Create a DataFrame
df = pd.DataFrame({'filepath': filepaths, 'label': labels, 
                   'class_name': ['Ripe' if l == 1 else 'Unripe' for l in labels]})

print(f"Total images found: {len(df)}")

# Display dataset summary
print("\nDataset Summary:")
print(df.head())

# Class Distribution
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='class_name', palette='Set2')
plt.title("Class Distribution")
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()

print("\nClass Counts:")
print(df['class_name'].value_counts())

In [ ]:
# Show sample images
plt.figure(figsize=(12, 6))
for i in range(10):
    idx = random.randint(0, len(df)-1)
    img_path = df.iloc[idx]['filepath']
    label_name = df.iloc[idx]['class_name']
    
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    plt.subplot(2, 5, i+1)
    plt.imshow(img)
    plt.title(label_name)
    plt.axis('off')
plt.tight_layout()
plt.show()

## 04 Exploratory Data Analysis
- Visualize image dimensions to understand variance in aspect ratios and sizes.
- Check for corrupted or unreadable images.

In [ ]:
# Check Image Dimensions and Corruptions
heights = []
widths = []
corrupted = []

for idx, row in df.iterrows():
    img_path = row['filepath']
    try:
        img = cv2.imread(img_path)
        if img is None:
            corrupted.append(img_path)
        else:
            heights.append(img.shape[0])
            widths.append(img.shape[1])
    except Exception as e:
        corrupted.append(img_path)

print(f"Total Corrupted/Missing Images: {len(corrupted)}")
if len(corrupted) > 0:
    print("Removing corrupted images from dataframe...")
    df = df[~df['filepath'].isin(corrupted)]

# Plot Image Dimensions
plt.figure(figsize=(10, 5))
plt.scatter(widths, heights, alpha=0.5, color='purple')
plt.title("Image Dimensions (Width vs Height)")
plt.xlabel("Width (Pixels)")
plt.ylabel("Height (Pixels)")
plt.grid(True)
plt.show()

print(f"Average Image Dimension: {int(np.mean(widths))}x{int(np.mean(heights))}")

**Observation:** Images vary significantly in dimensions, necessitating resizing during preprocessing to feed them into a fixed-input CNN.

## 05 Data Preprocessing
- Create reusable preprocessing functions.
- Train, validation, and test split.
- Define Image Dimensions for CNN (e.g., 128x128).

In [ ]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 16

# Convert labels to string for ImageDataGenerator
df['label_str'] = df['label'].astype(str)

# Train, Validation, Test Split
# 70% Train, 15% Validation, 15% Test
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=SEED, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=SEED, stratify=temp_df['label'])

print(f"Training set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")
print(f"Test set size: {len(test_df)}")

## 06 Data Augmentation
- Apply data augmentation using `ImageDataGenerator` on the training set to improve generalization and prevent overfitting.
- Validation and Test sets should only be normalized (rescaled), not augmented.

In [ ]:
# Training Generator with Augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,           # Normalize pixel values
    rotation_range=30,        # Randomly rotate images
    width_shift_range=0.2,    # Randomly shift images horizontally
    height_shift_range=0.2,   # Randomly shift images vertically
    shear_range=0.2,          # Randomly shear images
    zoom_range=0.2,           # Randomly zoom in
    horizontal_flip=True,     # Randomly flip images horizontally
    fill_mode='nearest'       # Strategy for filling in newly created pixels
)

# Validation and Test Generators (Only Rescaling)
val_test_datagen = ImageDataGenerator(rescale=1./255)

# Create Generators
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='filepath',
    y_col='label_str',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    seed=SEED
)

val_generator = val_test_datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col='filepath',
    y_col='label_str',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='filepath',
    y_col='label_str',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

In [ ]:
# Visualize Augmented Samples
augmented_images, augmented_labels = next(train_generator)

plt.figure(figsize=(12, 6))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.imshow(augmented_images[i])
    label = 'Ripe' if augmented_labels[i] == 1 else 'Unripe'
    plt.title(label)
    plt.axis('off')
plt.suptitle("Augmented Training Samples", fontsize=16)
plt.tight_layout()
plt.show()

**Observation:** The data generator successfully applies random transformations and normalizes the pixel values between 0 and 1.

## 07 CNN Architecture
Building a custom CNN from scratch with:
- Multiple Convolutional layers for feature extraction.
- MaxPooling layers for spatial downsampling.
- Batch Normalization for training stability.
- Dropout layers for regularization (preventing overfitting).
- Dense (Fully Connected) layers.
- Sigmoid output for binary classification.

In [ ]:
def build_cnn_model(input_shape):
    model = Sequential([
        # Block 1
        Conv2D(32, (3, 3), activation='relu', input_shape=input_shape, padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        
        # Block 2
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        
        # Block 3
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        
        # Block 4
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        
        # Flatten and Dense Layers
        Flatten(),
        Dense(256, activation='relu'),
        Dropout(0.5), # Regularization
        Dense(1, activation='sigmoid') # Binary classification output
    ])
    
    # Compile the model
    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    
    return model

input_shape = (IMG_SIZE[0], IMG_SIZE[1], 3)
cnn_model = build_cnn_model(input_shape)
cnn_model.summary()

## 08 Model Training
- Implement EarlyStopping to halt training when validation loss stops improving.
- Implement ModelCheckpoint to save the best model weights.
- Train the model and track history.

In [ ]:
# Define Callbacks
checkpoint_path = "best_cnn_model.h5"
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint(checkpoint_path, monitor='val_accuracy', save_best_only=True, verbose=1)
]

# Train Model
EPOCHS = 2

history = cnn_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=callbacks
)

## 09 Model Evaluation
Evaluate the trained CNN on the unseen Test set using multiple metrics:
- Accuracy, Precision, Recall, F1 Score
- Confusion Matrix and Classification Report
- ROC and Precision-Recall Curves

In [ ]:
# Load the best model weights
cnn_model.load_weights(checkpoint_path)

# Predict on Test Set
test_loss, test_accuracy = cnn_model.evaluate(test_generator)
print(f"\nTest Accuracy: {test_accuracy:.4f}")
print(f"Test Loss: {test_loss:.4f}")

# Get probabilities and predicted classes
y_prob = cnn_model.predict(test_generator)
y_pred = (y_prob > 0.5).astype(int).flatten()
y_true = test_generator.classes

# Evaluation Metrics
print("\n--- Classification Report ---")
print(classification_report(y_true, y_pred, target_names=class_names))

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
roc_auc = roc_auc_score(y_true, y_prob)

print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix (CNN Test Set)")
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

## 10 Training Visualization
Plot Training vs Validation Accuracy and Loss to analyze learning progress and check for overfitting.

In [ ]:
# Plot Accuracy and Loss
plt.figure(figsize=(14, 5))

# Accuracy Plot
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
plt.title('Training vs Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Loss Plot
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
plt.title('Training vs Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# ROC and Precision-Recall Curves
fpr, tpr, _ = roc_curve(y_true, y_prob)
precision, recall, _ = precision_recall_curve(y_true, y_prob)

plt.figure(figsize=(14, 5))

# ROC Curve
plt.subplot(1, 2, 1)
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.2f})', color='darkorange', lw=2)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.title('Receiver Operating Characteristic (ROC)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.grid(True)

# PR Curve
plt.subplot(1, 2, 2)
plt.plot(recall, precision, color='purple', lw=2)
plt.title('Precision-Recall Curve')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.grid(True)

plt.tight_layout()
plt.show()

## 11 Model Analysis

- **Overfitting vs Underfitting:** Analyzing the training curves, we can observe whether the training accuracy and validation accuracy diverge. By utilizing Dropout layers (0.5) and Data Augmentation, the model is penalized for relying too heavily on specific pathways, effectively bridging the gap between training and validation accuracy and preventing significant overfitting.
- **Impact of Data Augmentation:** The relatively small size of the dataset poses a risk for complex CNNs to memorize training samples. Augmentation introduces variance (rotation, shifts, flips) on the fly, forcing the CNN to learn invariant spatial features representing the "concept" of a tomato's ripeness, regardless of orientation or size.
- **Batch Normalization:** Implementing Batch Normalization after convolutional layers stabilized the learning process, allowing the model to converge faster and smoother by standardizing the inputs to subsequent layers.

## 12 Conclusions

**Key Findings:**
1. Transitioning to Deep Learning eliminated the need for manual feature engineering. The custom CNN successfully learned hierarchical spatial features directly from the raw pixel data.
2. The CNN achieves highly competitive performance on the test set, demonstrating robust classification capabilities between ripe and unripe tomatoes.
3. Techniques such as Data Augmentation, Dropout, and Early Stopping were critical in managing overfitting and ensuring generalizability on unseen data.

**Dataset Limitations & Challenges:**
- **Dataset Size:** Deep Learning models thrive on massive amounts of data. The current dataset is relatively small, which can limit the depth of the architecture we can effectively train without aggressive overfitting.
- **Background Variations:** High variability in background textures (greenhouses, soil, hands) requires robust feature extraction.

**Improvements for Future Work (Upcoming Phases):**
- **Transfer Learning:** Integrating architectures like ResNet, EfficientNet, or MobileNet to leverage pre-trained weights, likely boosting accuracy and reducing training time.
- **Object Detection (YOLO):** Upgrading from simple classification to bounding box detection to identify and localize multiple tomatoes simultaneously in complex real-world frames.
- **Segmentation:** Implementing semantic/instance segmentation to identify the exact pixel boundaries of tomatoes, which is crucial for precise robotic arm grasping.
- **Generative AI:** Using GANs or Diffusion models to generate synthetic tomato images and augment the dataset further.
- **Agentic AI:** Developing an intelligent agent to continuously monitor crop status and automate agricultural workflows.